In [21]:
import pandas as pd
import numpy as np

In this notebook, we will prepare our raw merge 2 file for regression performance. We start by importing the file as is.

In [22]:
merge2 = pd.read_csv("csv_data/MERGE2.csv")

# let's see what columns we have
merge2.columns

Index(['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'sic', 'datadate',
       'gvkey', 'conm', 'tic', 'fyear', 'at', 'ceq', 'dltt', 'lse', 'ni',
       'revt', 'xrd', 'csho', 'prcc_f', 'sich', 'mkt_cap', 'industry',
       'boardid', 'companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority', 'UG', 'top20_ug', 'MBA', 'top20_mba',
       'PhD', 'top20_phd', 'MD', 'top20_md', 'Master's', 'top20_masters',
       'dob', 'gender', 'diversitynetworklabel'],
      dtype='object')

In [23]:
merge2.isna().sum()

costat                     0
curcd                      0
datafmt                    0
indfmt                     0
consol                     0
sic                        0
datadate                   0
gvkey                      0
conm                       0
tic                        0
fyear                      0
at                         0
ceq                        0
dltt                       0
lse                        0
ni                         0
revt                       0
xrd                       87
csho                       0
prcc_f                     0
sich                       3
mkt_cap                    0
industry                   0
boardid                    0
companyid                  0
datestartrole              0
directorid                 0
directorname               0
companyname                0
rolename                   0
dateendrole                0
datestartroleflag          0
dateendroleflag            0
seniority                  0
UG            

We need to get some additional data from compustat. We get our tickers for this pull below:

In [24]:
%run processes/ticker_txt_conversion.py

We gather retained earnings from Compustat and bring them in:

In [25]:
retained_earnings = pd.read_csv("csv_data/retained_earnings.csv")

In [26]:
retained_earnings['fyear'] = pd.to_datetime(retained_earnings['datadate']).dt.year

merge2 = merge2.merge(
    retained_earnings[['gvkey', 'fyear', 're']],
    on=['gvkey', 'fyear'],
    how='left'
)

print(merge2.shape)
print(merge2['re'].isna().sum())

(1032, 48)
0


Now we are going to compute volatility:

In [28]:
volatility = pd.read_csv("csv_data/volatility.csv")
volatility['date'] = pd.to_datetime(volatility['DlyCalDt'])
volatility['fyear'] = volatility['date'].dt.year

vol_annual = volatility.groupby(['Ticker', 'fyear'])['DlyRet'].std().reset_index()
vol_annual.columns = ['tic', 'fyear', 'volatility']

merge2 = merge2.merge(
    vol_annual[['tic', 'fyear', 'volatility']],
    on=['tic', 'fyear'],
    how='left'
)

print(merge2.shape)
print(merge2['volatility'].isna().sum())

(1032, 49)
35


In [29]:
baseline_reg = merge2.copy()

In [30]:
# DEPENDENT VARIABLE
baseline_reg["roa"] = baseline_reg["ni"] / baseline_reg["at"]
baseline_reg['adjusted_roa'] = baseline_reg['roa'] - baseline_reg.groupby('fyear')['roa'].transform('mean')

# KING ET AL relevant controlls
baseline_reg['firm_size'] = np.log(baseline_reg['at'])
baseline_reg["equity_capital"] = baseline_reg["ceq"] / baseline_reg["at"]
charter_ratio = baseline_reg['mkt_cap'] / baseline_reg['ceq']
baseline_reg['charter_value'] = np.where(charter_ratio > 0, np.log(charter_ratio), np.nan)# volatility
baseline_reg["retained_earnings"] = baseline_reg["re"] / baseline_reg["at"]
baseline_reg["volatility"] = baseline_reg["volatility"]
# macro conditions

# my added controls:
# R&D
baseline_reg['xrd'] = baseline_reg['xrd'].fillna(0)
baseline_reg['rd_intensity'] = baseline_reg['xrd'] / baseline_reg['at']
#age and gender
baseline_reg["ceo_age"] = baseline_reg["fyear"] - pd.to_datetime(baseline_reg['dob']).dt.year
baseline_reg['ceo_gender'] = (baseline_reg['gender'] == 'Female').astype(int)

print(baseline_reg['charter_value'].isna().sum())


40


/Users/thefleok/Desktop/GitHub/QMSS_CEO_Thesis/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [32]:
baseline_reg.isna().sum()

costat                     0
curcd                      0
datafmt                    0
indfmt                     0
consol                     0
sic                        0
datadate                   0
gvkey                      0
conm                       0
tic                        0
fyear                      0
at                         0
ceq                        0
dltt                       0
lse                        0
ni                         0
revt                       0
xrd                        0
csho                       0
prcc_f                     0
sich                       3
mkt_cap                    0
industry                   0
boardid                    0
companyid                  0
datestartrole              0
directorid                 0
directorname               0
companyname                0
rolename                   0
dateendrole                0
datestartroleflag          0
dateendroleflag            0
seniority                  0
UG            

Let's construct our core baseline regression data.